# 06_group_pipeline. 그룹별 완전분리 + 피처 확장 재구조화 (독립 신규 파이프라인)

**배경**: 지금까지의 파이프라인(50개 curated 피처 + CatBoost/MLP 하이브리드, 확정 v5, LB 0.629551)에서 시도 가능한 저비용 레버(피처 rolling/diff, CatBoost/MLP 재튜닝, FICR 결정이론 분위수 앙상블, seed 앙상블 크기)를 모두 시도했고 전부 기각됐다(`HANDOFF.md`, `reports/05_tuning.md` 25~28절). 같은 대회를 다루는 별도 파이프라인(`D:\공모전\wind-forecast_Competition`)은 **그룹별 완전분리 모델 + 179개 피처**로 Public 0.63886을 기록해 우리보다 0.0093 높다.

민석님 지시: "seed 앙상블 해보고 안되면 외부 파이프라인처럼 가보자, 새 파일로 시작해." → 기존 01~04·05_tuning_*·train/inference는 건드리지 않고 이 노트북 하나에 구조 재작업을 담는다.

**핵심 설계 원칙**:
1. **번들 변경 디컨파운드를 사전에 설계** — 구조 변경(완전분리)과 피처 변경(확장)을 동시에 바꾸면 사후에 순효과를 못 가르므로, 처음부터 A/B/C/D 2×2 비교로 설계한다(5절).
2. **leakage-guard 원칙 엄수** — 외부 파이프라인의 lag/lead 코드가 발표분 경계를 지켰는지 불확실하므로, 우리는 반드시 `data_available_kst_dtm` 경계 안에서만 시간 파생을 계산한다.
3. **g3 완전분리 안전장치** — 05_tuning_3 7-1에서 기존 50피처로는 g3 단독학습이 -0.0126 손해였다. 확장 피처로도 못 메우면 g3만 통합(3그룹 공동학습)으로 폴백한다(민석님 결정).
4. **하이퍼파라미터는 재탐색하지 않는다** — 기존에 검증된 값(v5 설정)을 그대로 시작점으로 써서 "구조+피처가 유망한지"부터 스크린하고, 유망하면 그 다음에 재튜닝한다.

**미리 밝히는 스코프 축소**: 외부 1위 피처(`pc_pred` roll5, 파워커브 SCADA 회귀)는 이전 17절에서 전체 설계까지 했으나 실패로 결론나 코드가 삭제됐다. 이번 빌드에서 되살리는 건 별도 서브프로젝트급이라 **이번엔 재현하지 않는다** — 검증된 다른 미사용 컬럼(상층대기·구름·복사·강수·경계층 등)에 집중한다.

## 0. 셋업

In [1]:
# 0-1. 셋업 + 공통 헬퍼(03_features.ipynb 패턴 재사용, self-contained)
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
import torch

SEED = 42
np.random.seed(SEED)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "data").exists(), "REPO_ROOT를 찾지 못했습니다."

sys.path.insert(0, str(REPO_ROOT))
from src.metric import metric, TARGET_COLS, CAPACITY_KWH
from src import nn as mlp_nn

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
pd.set_option("display.max_columns", 100)

# 02_eda 2-1절에서 확정된 그룹-격자 가중치 (03_features.ipynb 0-1과 동일)
GROUP_GRID_WEIGHTS = {
    "kpx_group_1": {"gfs": {5: 1.0}, "ldaps": {5: 2 / 3, 6: 1 / 3}},
    "kpx_group_2": {"gfs": {5: 1.0}, "ldaps": {6: 0.5, 11: 0.5}},
    "kpx_group_3": {"gfs": {5: 1.0}, "ldaps": {12: 0.8, 6: 0.2}},
}


def group_weighted_uv(df, group, source, u_var, v_var):
    weights = GROUP_GRID_WEIGHTS[group][source]
    u_total, v_total = 0.0, 0.0
    for grid_id, weight in weights.items():
        u_total = u_total + df[f"{source}_g{grid_id}_{u_var}"] * weight
        v_total = v_total + df[f"{source}_g{grid_id}_{v_var}"] * weight
    return u_total, v_total


def group_weighted_scalar(df, group, source, var):
    weights = GROUP_GRID_WEIGHTS[group][source]
    total = 0.0
    for grid_id, weight in weights.items():
        total = total + df[f"{source}_g{grid_id}_{var}"] * weight
    return total


def wind_speed(u, v):
    return np.sqrt(u ** 2 + v ** 2)


def wind_direction(u, v):
    return (270 - np.degrees(np.arctan2(v, u))) % 360


print("python:", sys.executable, "| torch:", torch.__version__)
print("TARGET_COLS:", TARGET_COLS, "| CAPACITY_KWH:", CAPACITY_KWH)

python: d:\공모전\wind_forecast\venv\Scripts\python.exe | torch: 2.13.0+cpu
TARGET_COLS: ['kpx_group_1', 'kpx_group_2', 'kpx_group_3'] | CAPACITY_KWH: {'kpx_group_1': 21600, 'kpx_group_2': 21600, 'kpx_group_3': 21000}


In [2]:
# 0-2. 원본 데이터(base_wide/base_agg) + 기존 v2 피처(비교 기준선) 로드
train_raw = pd.read_parquet(PROCESSED_DIR / "train_base_wide.parquet")
test_raw = pd.read_parquet(PROCESSED_DIR / "test_base_wide.parquet")
train_agg = pd.read_parquet(PROCESSED_DIR / "train_base_agg.parquet")
test_agg = pd.read_parquet(PROCESSED_DIR / "test_base_agg.parquet")
train_v2 = pd.read_parquet(PROCESSED_DIR / "train_features_v2.parquet")  # baseline 50피처(비교 기준 A/D)

assert (train_raw["forecast_kst_dtm"].values == train_agg["forecast_kst_dtm"].values).all()
assert (test_raw["forecast_kst_dtm"].values == test_agg["forecast_kst_dtm"].values).all()
assert (train_raw["forecast_kst_dtm"].values == train_v2["forecast_kst_dtm"].values).all()
assert train_raw["forecast_kst_dtm"].is_monotonic_increasing
assert test_raw["forecast_kst_dtm"].is_monotonic_increasing


def fix_ldaps_quality(df):
    """03_features.ipynb와 동일: RH 100% 클리핑 + 발표분 내 결측 선형보간(경계 밖은 안 채움 -> 누수 아님)."""
    df = df.sort_values("forecast_kst_dtm").reset_index(drop=True).copy()
    rh_cols = [c for c in df.columns if c.startswith("ldaps_g") and c.endswith("heightAboveGround_2_r")]
    df[rh_cols] = df[rh_cols].clip(upper=100)
    ldaps_cols = [c for c in df.columns if c.startswith("ldaps_g")]
    if df[ldaps_cols].isna().any().any():
        df[ldaps_cols] = df.groupby("ldaps_data_available_kst_dtm")[ldaps_cols].transform(
            lambda s: s.interpolate(method="linear", limit_area="inside")
        )
    return df


print("보간 전 결측(train/test):", train_raw.filter(regex="^ldaps_g").isna().sum().sum(), test_raw.filter(regex="^ldaps_g").isna().sum().sum())
train_raw = fix_ldaps_quality(train_raw)
test_raw = fix_ldaps_quality(test_raw)
print("보간 후 결측(train/test):", train_raw.filter(regex="^ldaps_g").isna().sum().sum(), test_raw.filter(regex="^ldaps_g").isna().sum().sum())

BASELINE_COLS = [c for c in train_v2.columns if c not in {"forecast_kst_dtm", "forecast_id", *TARGET_COLS}]
print("train_raw:", train_raw.shape, "| test_raw:", test_raw.shape, "| baseline 피처:", len(BASELINE_COLS))

FOLDS = [
    {"name": "fold1", "train_start": "2022-01-01 01:00:00", "train_end": "2023-07-01 00:00:00", "valid_end": "2024-01-01 00:00:00"},
    {"name": "fold2", "train_start": "2022-01-01 01:00:00", "train_end": "2024-01-01 00:00:00", "valid_end": "2024-07-01 00:00:00"},
    {"name": "fold3", "train_start": "2022-01-01 01:00:00", "train_end": "2024-07-01 00:00:00", "valid_end": "2025-01-01 00:00:00"},
]


def fold_frames(df, fold):
    dtm = df["forecast_kst_dtm"]
    ts, te, ve = pd.Timestamp(fold["train_start"]), pd.Timestamp(fold["train_end"]), pd.Timestamp(fold["valid_end"])
    vs = te + pd.Timedelta(hours=1)
    tr = df[(dtm >= ts) & (dtm <= te)].reset_index(drop=True)
    va = df[(dtm >= vs) & (dtm <= ve)].reset_index(drop=True)
    return tr, va


for fold in FOLDS:
    tr, va = fold_frames(train_v2, fold)
    print(f"{fold['name']}: train {tr.shape}, valid {va.shape}")

보간 전 결측(train/test): 0 752
보간 후 결측(train/test): 0 0
train_raw: (26304, 801) | test_raw: (8760, 799) | baseline 피처: 50
fold1: train (13104, 54), valid (4416, 54)
fold2: train (17520, 54), valid (4368, 54)
fold3: train (21888, 54), valid (4416, 54)


## 1. 신규 피처 A — 미사용 원본 컬럼의 단일시점 파생

`data/processed/train_base_wide.parquet`(GFS 9격자+LDAPS 16격자)엔 있지만 기존 50개 피처엔 없는 원본 변수들을 물리적 근거로 파생한다. 카테고리별 근거(`wind-domain-features` 스킬):

- **상층대기(700/500hPa, 850hPa 습도)**: 산 정상(가덕산) 자유대기 대표성(850hPa는 이미 사용중, 700/500으로 확장). `gh500`(500hPa 지위고도)은 종관 기압골/능 지표
- **안정도·결빙**(icing_score, lapse_850_500, VRATE, prmsl): 05_tuning_2 20절 코드 재사용(미실행이었음, 여기서 검증). 대기 안정도가 상층운동량의 지상 혼합에 영향
- **경계층 풍속(PBL u/v)**: 지상 10m와 별개로 경계층 내 대표 풍속 — 혼합층 운동량의 직접 지표
- **구름·복사**(lcc/mcc/hcc/tcc, dswrf/dlwrf 등): 지표 가열이 대류혼합층 발달과 난류에 영향(대류 경계층 이론)
- **강수**(prate/tp 등): 전선 통과(윈드램프 이벤트) 대리 지표
- **50m 최대/최소 풍속**: 시어·돌풍 강도(기존엔 최대-최소 차이만 사용, 절댓값 추가)
- **비습 기반 가온도 밀도 보정**: 기존 건조공기 밀도(ρ=p/RT)를 가온도(Tv=T(1+0.61q))로 정밀화(IEC 61400-12-1과 일치하는 표준 보정)
- **격자 std**(GFS 주요 변수 공간표준편차): 예보 불확실성 대리(`wind-domain-features` 5절 근거)

**제외한 것과 이유**: `heightAboveGround_5_XBLWS/YBLWS`("5m 경계층 바람")는 실측 검증 결과 평균 0.21m/s로 10m 풍속(평균 4.8m/s)의 약 1/23에 불과 — 로그 풍속 프로파일상 5m/10m 풍속비가 이렇게 작을 수 없어(물리적으로 90% 안팎이어야 함) **이 필드의 의미를 정확히 알 수 없다고 판단, 제외**한다. `surface_0_h`/`surface_0_lsm`(지형고도/육해마스크)도 그룹별로 시간에 따라 값이 전혀 변하지 않아(nunique=1, 격자가 고정이라 상수) 학습에 정보가 없어 제외한다.

In [3]:
# 1-1. GFS 단일시점 파생 (격자5, 전 그룹 공유 -> kpx_group_1 가중치로 대표 계산)
HUB_HEIGHT_M = 117.0  # info.xlsx Hub Height


def build_gfs_atmos_features(raw_df):
    feats = pd.DataFrame({"forecast_kst_dtm": raw_df["forecast_kst_dtm"]})
    g = "kpx_group_1"  # GFS 가중치는 전 그룹 동일(격자5=1.0)

    t2m_c = group_weighted_scalar(raw_df, g, "gfs", "heightAboveGround_2_2t") - 273.15
    r2m = group_weighted_scalar(raw_df, g, "gfs", "heightAboveGround_2_2r").clip(upper=100.0)
    feats["gfs_icing_score"] = np.exp(-(((t2m_c - (-3.0)) / 4.0) ** 2)) * ((r2m - 80.0) / 20.0).clip(0.0, 1.0)

    u700, v700 = group_weighted_uv(raw_df, g, "gfs", "isobaricInhPa_700_u", "isobaricInhPa_700_v")
    feats["gfs_ws700hpa"] = wind_speed(u700, v700)
    wd700 = wind_direction(u700, v700)
    feats["gfs_wd700hpa_sin"] = np.sin(np.radians(wd700))
    feats["gfs_wd700hpa_cos"] = np.cos(np.radians(wd700))

    u500, v500 = group_weighted_uv(raw_df, g, "gfs", "isobaricInhPa_500_u", "isobaricInhPa_500_v")
    feats["gfs_ws500hpa"] = wind_speed(u500, v500)
    feats["gfs_gh500"] = group_weighted_scalar(raw_df, g, "gfs", "isobaricInhPa_500_gh")
    feats["gfs_r850hpa"] = group_weighted_scalar(raw_df, g, "gfs", "isobaricInhPa_850_r").clip(0, 100)
    feats["gfs_lapse_850_500"] = (
        group_weighted_scalar(raw_df, g, "gfs", "isobaricInhPa_850_t")
        - group_weighted_scalar(raw_df, g, "gfs", "isobaricInhPa_500_t")
    )
    feats["gfs_prmsl"] = group_weighted_scalar(raw_df, g, "gfs", "meanSea_0_prmsl")
    feats["gfs_vrate"] = group_weighted_scalar(raw_df, g, "gfs", "planetaryBoundaryLayer_0_VRATE")

    u_pbl, v_pbl = group_weighted_uv(raw_df, g, "gfs", "planetaryBoundaryLayer_0_u", "planetaryBoundaryLayer_0_v")
    feats["gfs_ws_pbl"] = wind_speed(u_pbl, v_pbl)

    t2d = group_weighted_scalar(raw_df, g, "gfs", "heightAboveGround_2_2d")
    feats["gfs_dewpoint_depression"] = (t2m_c + 273.15) - t2d

    q2m = group_weighted_scalar(raw_df, g, "gfs", "heightAboveGround_2_2sh")
    tv = (t2m_c + 273.15) * (1 + 0.61 * q2m)
    sp = group_weighted_scalar(raw_df, g, "gfs", "surface_0_sp")
    feats["gfs_rho_tv"] = sp / (287.05 * tv)  # 가온도 보정 밀도(IEC 61400-12-1 표준 정밀화)

    feats["gfs_lcc"] = group_weighted_scalar(raw_df, g, "gfs", "lowCloudLayer_0_lcc")
    feats["gfs_mcc"] = group_weighted_scalar(raw_df, g, "gfs", "middleCloudLayer_0_mcc")
    feats["gfs_hcc"] = group_weighted_scalar(raw_df, g, "gfs", "highCloudLayer_0_hcc")
    feats["gfs_tcc"] = group_weighted_scalar(raw_df, g, "gfs", "atmosphere_0_tcc")
    feats["gfs_dswrf"] = group_weighted_scalar(raw_df, g, "gfs", "surface_0_dswrf")
    feats["gfs_dlwrf"] = group_weighted_scalar(raw_df, g, "gfs", "surface_0_dlwrf")
    feats["gfs_prate"] = group_weighted_scalar(raw_df, g, "gfs", "surface_0_prate")
    feats["gfs_tp"] = group_weighted_scalar(raw_df, g, "gfs", "surface_0_tp")
    return feats


GFS_ATMOS_COLS = [c for c in build_gfs_atmos_features(train_raw.head(2)).columns if c != "forecast_kst_dtm"]
train_gfs_atmos = build_gfs_atmos_features(train_raw)
test_gfs_atmos = build_gfs_atmos_features(test_raw)
print("GFS 신규 단일시점 피처:", len(GFS_ATMOS_COLS), "개")
print("train 결측:", train_gfs_atmos[GFS_ATMOS_COLS].isna().sum().sum(), "| test 결측:", test_gfs_atmos[GFS_ATMOS_COLS].isna().sum().sum())
train_gfs_atmos[GFS_ATMOS_COLS].describe().T[["min", "mean", "max"]]

GFS 신규 단일시점 피처: 21 개
train 결측: 0 | test 결측: 0


,min,mean,max
gfs_icing_score,0.000000,0.035611,0.957490
gfs_ws700hpa,0.104995,11.850140,45.095540
gfs_wd700hpa_sin,-1.000000,-0.668805,0.999995
gfs_wd700hpa_cos,-1.000000,0.103215,1.000000
gfs_ws500hpa,0.067553,19.486161,71.030052
gfs_gh500,5055.260700,5663.376176,5971.515600
gfs_r850hpa,2.500000,61.714815,100.000000
gfs_lapse_850_500,1.683070,21.710246,34.213540
gfs_prmsl,98885.920000,101651.598115,103670.940000
gfs_vrate,0.000000,4099.635193,67000.000000


In [4]:
# 1-2. LDAPS 그룹별 단일시점 파생 (XBLWS/YBLWS, h, lsm 제외 — 마크다운 근거 참조)
def build_ldaps_group_features(raw_df):
    feats = pd.DataFrame({"forecast_kst_dtm": raw_df["forecast_kst_dtm"]})
    for g in TARGET_COLS:
        t_c = group_weighted_scalar(raw_df, g, "ldaps", "heightAboveGround_2_t") - 273.15
        r2m = group_weighted_scalar(raw_df, g, "ldaps", "heightAboveGround_2_r").clip(upper=100.0)
        feats[f"{g}_ldaps_icing_score"] = np.exp(-(((t_c - (-3.0)) / 4.0) ** 2)) * ((r2m - 80.0) / 20.0).clip(0.0, 1.0)

        u_max, v_max = group_weighted_uv(raw_df, g, "ldaps", "heightAboveGround_50_50MUmax", "heightAboveGround_50_50MVmax")
        feats[f"{g}_ldaps_ws50_max"] = wind_speed(u_max, v_max)
        u_min, v_min = group_weighted_uv(raw_df, g, "ldaps", "heightAboveGround_50_50MUmin", "heightAboveGround_50_50MVmin")
        feats[f"{g}_ldaps_ws50_min"] = wind_speed(u_min, v_min)

        q2 = group_weighted_scalar(raw_df, g, "ldaps", "heightAboveGround_2_q")
        tv = (t_c + 273.15) * (1 + 0.61 * q2)
        sp = group_weighted_scalar(raw_df, g, "ldaps", "surface_0_sp")
        feats[f"{g}_ldaps_rho_tv"] = sp / (287.05 * tv)

        feats[f"{g}_ldaps_prmsl"] = group_weighted_scalar(raw_df, g, "ldaps", "meanSea_0_prmsl")
        feats[f"{g}_ldaps_lcc"] = group_weighted_scalar(raw_df, g, "ldaps", "etc_0_lcc")
        feats[f"{g}_ldaps_mcc"] = group_weighted_scalar(raw_df, g, "ldaps", "etc_0_mcc")
        feats[f"{g}_ldaps_hcc"] = group_weighted_scalar(raw_df, g, "ldaps", "etc_0_hcc")
        feats[f"{g}_ldaps_vlcdc"] = group_weighted_scalar(raw_df, g, "ldaps", "etc_0_VLCDC")
        feats[f"{g}_ldaps_ndnsw"] = group_weighted_scalar(raw_df, g, "ldaps", "surface_0_NDNSW")
        feats[f"{g}_ldaps_ndnlw"] = group_weighted_scalar(raw_df, g, "ldaps", "surface_0_NDNLW")
        feats[f"{g}_ldaps_swtot"] = (
            group_weighted_scalar(raw_df, g, "ldaps", "heightAboveGround_2_SWDIR")
            + group_weighted_scalar(raw_df, g, "ldaps", "heightAboveGround_2_SWDIF")
        )
        feats[f"{g}_ldaps_precip_total"] = (
            group_weighted_scalar(raw_df, g, "ldaps", "surface_0_avg_lsprate")
            + group_weighted_scalar(raw_df, g, "ldaps", "surface_0_lssrate")
        )
        feats[f"{g}_ldaps_snow_total"] = (
            group_weighted_scalar(raw_df, g, "ldaps", "surface_0_snol")
            + group_weighted_scalar(raw_df, g, "ldaps", "surface_0_SNOM")
        )
    return feats


LDAPS_GROUP_COLS = [c for c in build_ldaps_group_features(train_raw.head(2)).columns if c != "forecast_kst_dtm"]
train_ldaps_group = build_ldaps_group_features(train_raw)
test_ldaps_group = build_ldaps_group_features(test_raw)
print("LDAPS 그룹별 신규 단일시점 피처:", len(LDAPS_GROUP_COLS), "개(3그룹 합산)")
print("train 결측:", train_ldaps_group[LDAPS_GROUP_COLS].isna().sum().sum(), "| test 결측:", test_ldaps_group[LDAPS_GROUP_COLS].isna().sum().sum())

LDAPS 그룹별 신규 단일시점 피처: 42 개(3그룹 합산)
train 결측: 0 | test 결측: 0


In [5]:
# 1-3. 격자 std(공간 불균일성 = 예보 불확실성 대리, GFS 주요 변수만 저비용 시작)
def build_grid_std_features(agg_df):
    feats = pd.DataFrame({"forecast_kst_dtm": agg_df["forecast_kst_dtm"]})
    feats["gfs_grid_std_ws10m"] = np.sqrt(
        agg_df["gfs_std_heightAboveGround_10_10u"] ** 2 + agg_df["gfs_std_heightAboveGround_10_10v"] ** 2
    )
    feats["gfs_grid_std_t2m"] = agg_df["gfs_std_heightAboveGround_2_2t"]
    return feats


GRID_STD_COLS = ["gfs_grid_std_ws10m", "gfs_grid_std_t2m"]
train_grid_std = build_grid_std_features(train_agg)
test_grid_std = build_grid_std_features(test_agg)
print("격자 std 신규 피처:", len(GRID_STD_COLS), "개, 결측:", train_grid_std[GRID_STD_COLS].isna().sum().sum())

# 1절 단일시점 신규 피처 조립
SINGLE_TS_NEW_COLS = GFS_ATMOS_COLS + LDAPS_GROUP_COLS + GRID_STD_COLS
train_single_new = train_gfs_atmos.merge(train_ldaps_group, on="forecast_kst_dtm").merge(train_grid_std, on="forecast_kst_dtm")
test_single_new = test_gfs_atmos.merge(test_ldaps_group, on="forecast_kst_dtm").merge(test_grid_std, on="forecast_kst_dtm")
assert (train_single_new["forecast_kst_dtm"].values == train_raw["forecast_kst_dtm"].values).all()
assert (test_single_new["forecast_kst_dtm"].values == test_raw["forecast_kst_dtm"].values).all()
print(f"\n1절 단일시점 신규 피처 총합: {len(SINGLE_TS_NEW_COLS)}개")
print("train 결측 총합:", train_single_new[SINGLE_TS_NEW_COLS].isna().sum().sum())
print("test 결측 총합:", test_single_new[SINGLE_TS_NEW_COLS].isna().sum().sum())

격자 std 신규 피처: 2 개, 결측: 0

1절 단일시점 신규 피처 총합: 65개
train 결측 총합: 0
test 결측 총합: 0


## 2. 신규 피처 B — 발표분 내 diff·rolling (핵심 변수만, trailing만 사용)

외부 파이프라인은 lag/lead(미래 시각 포함)까지 썼지만, 우리는 05_tuning_3 9-1에서 이미 leakage-safe로 검증한 **trailing만(diff, roll3/5 mean)** 방식을 그대로 쓴다 — lead는 leakage-guard상 허용되지만(같은 발표분 내 미래 시각은 이미 공개된 정보), 이 프로젝트에서 독립적으로 재검증한 적이 없어 더 보수적인 쪽을 기본값으로 유지한다.

**대상 5개 핵심 변수**(외부의 "소수의 기준 컬럼 선정" 방식 참고, 물리적 근거로 직접 선정):
- `gfs_ws850hpa`(기존 피처, permutation importance 1위였던 이력) — 자유대기 대표 풍속의 시간 변화
- `gfs_ws100m`(기존 피처, 허브고도 대표) — 순간 예보 오차를 시간 평활로 완화
- `{group}_ldaps_ws10m` ×3(기존 피처) — 그룹별 지역 풍속의 시간 변화(05_tuning_3 9절에서 50피처 파이프라인 기준으로는 기각됐으나, 완전분리+확장피처 맥락에서 재검증)
- `gfs_lapse_850_500`(신규, 1절) — 안정도 변화(전선 접근의 대리)
- `gfs_icing_score`(신규, 1절) — 결빙 조건의 지속성

**누수 안전**: `add_issuance_diff`(03_features)와 동일하게 `groupby(data_available_kst_dtm)` 안에서만 계산 — 발표분 경계를 넘지 않음.

In [6]:
# 2-1. 발표분 내 diff/rolling (trailing만) -- 누수 안전 검증 포함
ROLL_WINDOWS = (3, 5)


def add_trailing_features(df, agg_df, col_specs):
    """col_specs: [(col_name, avail_col_name)]. groupby(발표분) 안에서만 diff/roll 계산."""
    out = df.copy()
    new_cols = []
    for col, avail_col in col_specs:
        av = agg_df[avail_col].values
        out[f"{col}_diff1"] = out[col].groupby(av).transform(lambda x: x.diff())
        new_cols.append(f"{col}_diff1")
        for w in ROLL_WINDOWS:
            nm = f"{col}_roll{w}_mean"
            out[nm] = out[col].groupby(av).transform(lambda x: x.rolling(w, min_periods=1).mean())
            new_cols.append(nm)
    return out, new_cols


# 대상 데이터프레임 구성: 기존 v2 피처(ws850/ws100/ldaps_ws10m) + 1절 신규(lapse/icing)
CORE_TRAIL_SPECS = [
    ("gfs_ws850hpa", "gfs_data_available_kst_dtm"),
    ("gfs_ws100m", "gfs_data_available_kst_dtm"),
    ("kpx_group_1_ldaps_ws10m", "ldaps_data_available_kst_dtm"),
    ("kpx_group_2_ldaps_ws10m", "ldaps_data_available_kst_dtm"),
    ("kpx_group_3_ldaps_ws10m", "ldaps_data_available_kst_dtm"),
    ("gfs_lapse_850_500", "gfs_data_available_kst_dtm"),
    ("gfs_icing_score", "gfs_data_available_kst_dtm"),
]


def build_trailing_frame(v2_df, single_new_df, agg_df):
    base = v2_df[["forecast_kst_dtm", "gfs_ws850hpa", "gfs_ws100m",
                  "kpx_group_1_ldaps_ws10m", "kpx_group_2_ldaps_ws10m", "kpx_group_3_ldaps_ws10m"]].copy()
    base = base.merge(single_new_df[["forecast_kst_dtm", "gfs_lapse_850_500", "gfs_icing_score"]], on="forecast_kst_dtm")
    out, new_cols = add_trailing_features(base, agg_df, CORE_TRAIL_SPECS)
    return out[["forecast_kst_dtm"] + new_cols], new_cols


train_trailing, TRAILING_COLS = build_trailing_frame(train_v2, train_single_new, train_agg)
test_trailing, _ = build_trailing_frame(
    test_v2 if "test_v2" in dir() else pd.read_parquet(PROCESSED_DIR / "test_features_v2.parquet"),
    test_single_new, test_agg,
)

print(f"발표분 내 diff/rolling 신규 피처: {len(TRAILING_COLS)}개")

# 누수 검증: 블록 첫 시각 roll3==원값, diff1==NaN (05_tuning_3 9-1과 동일 검증 패턴)
blk_first = ~pd.Series(train_agg["gfs_data_available_kst_dtm"].values).duplicated().values
check_col = "gfs_ws850hpa"
merged_check = train_v2[["forecast_kst_dtm", check_col]].merge(train_trailing, on="forecast_kst_dtm")
print("블록 첫 시각 roll3==원값(누수없음):",
      np.allclose(merged_check.loc[blk_first, f"{check_col}_roll3_mean"], merged_check.loc[blk_first, check_col]))
print("diff1 결측(=발표분 첫 시각):", int(merged_check[f"{check_col}_diff1"].isna().sum()), "/ 블록수", int(blk_first.sum()))
print("전체 결측(diff1 제외 0이어야):", int(train_trailing[[c for c in TRAILING_COLS if "diff1" not in c]].isna().sum().sum()))

발표분 내 diff/rolling 신규 피처: 21개
블록 첫 시각 roll3==원값(누수없음): True
diff1 결측(=발표분 첫 시각): 1096 / 블록수 1096
전체 결측(diff1 제외 0이어야): 0


## 3. 최종 피처셋 조립 + leakage-guard 검증

In [7]:
# 3-1. 최종 피처셋 조립(baseline 50 + 신규 단일시점 + 신규 trailing)
train_features_aug = (
    train_v2[["forecast_kst_dtm", *TARGET_COLS] + BASELINE_COLS]
    .merge(train_single_new, on="forecast_kst_dtm")
    .merge(train_trailing, on="forecast_kst_dtm")
)
test_v2_full = pd.read_parquet(PROCESSED_DIR / "test_features_v2.parquet")
test_features_aug = (
    test_v2_full[["forecast_kst_dtm"] + BASELINE_COLS]
    .merge(test_single_new, on="forecast_kst_dtm")
    .merge(test_trailing, on="forecast_kst_dtm")
)

FEATURE_COLS_AUG = BASELINE_COLS + SINGLE_TS_NEW_COLS + TRAILING_COLS
assert len(FEATURE_COLS_AUG) == len(set(FEATURE_COLS_AUG)), "중복 피처명 존재"
assert set(train_features_aug.columns) - set(TARGET_COLS) - {"forecast_kst_dtm"} == set(test_features_aug.columns) - {"forecast_kst_dtm"} \
    or set(FEATURE_COLS_AUG) <= set(test_features_aug.columns), "train/test 피처 컬럼 불일치"

# leakage-guard 체크리스트
assert (train_features_aug["forecast_kst_dtm"].values == train_v2["forecast_kst_dtm"].values).all()
assert train_features_aug["forecast_kst_dtm"].is_monotonic_increasing
na_counts = train_features_aug[FEATURE_COLS_AUG].isna().sum()
na_nonzero = na_counts[na_counts > 0]
print(f"baseline({len(BASELINE_COLS)}) + 신규 단일시점({len(SINGLE_TS_NEW_COLS)}) + 신규 trailing({len(TRAILING_COLS)}) "
      f"= 총 {len(FEATURE_COLS_AUG)}개 피처")
print("결측 있는 컬럼(diff1류만 있어야 함, 발표분 첫 시각):")
print(na_nonzero)
print("\ntrain_features_aug:", train_features_aug.shape, "| test_features_aug:", test_features_aug.shape)

baseline(50) + 신규 단일시점(65) + 신규 trailing(21) = 총 136개 피처
결측 있는 컬럼(diff1류만 있어야 함, 발표분 첫 시각):
gfs_ws100m_diff_prev                 1096
kpx_group_1_ldaps_ws10m_diff_prev    1096
kpx_group_2_ldaps_ws10m_diff_prev    1096
kpx_group_3_ldaps_ws10m_diff_prev    1096
gfs_ws850hpa_diff1                   1096
gfs_ws100m_diff1                     1096
kpx_group_1_ldaps_ws10m_diff1        1096
kpx_group_2_ldaps_ws10m_diff1        1096
kpx_group_3_ldaps_ws10m_diff1        1096
gfs_lapse_850_500_diff1              1096
gfs_icing_score_diff1                1096
dtype: int64

train_features_aug: (26304, 140) | test_features_aug: (8760, 137)


## 4. 그룹별 완전분리 학습 인프라

CatBoost·MLP 둘 다 **그룹마다 완전히 독립적으로** 학습하는 헬퍼(하이브리드 A/D와 달리 group_id를 안 쓰고, 그 그룹 데이터만 필터링). `05_tuning_3` 7절의 `to_long_single`/`train_group_mlp` 패턴을 그대로 재사용(이번 노트북에서 self-contained로 재정의).

**하이퍼파라미터는 v5에서 검증된 값을 그대로 시작점으로 쓴다**(재탐색 안 함): CatBoost는 그룹별 동일(`iterations=2000, lr=0.05, τ=0.70`), MLP는 g1=(256,256)/drop0.10, g2=(384,384)/drop0.05, **g3=(256,256)/drop0.15(v5 통합 설정과 동일값을 완전분리 시작점으로 사용, 이후 유망하면 재탐색)**. `T_soft=0.035, lr=1e-3, wd=1e-4` 전 그룹 동일.

In [8]:
# 4-1. 완전분리 학습 헬퍼 (그룹 필터링만, group_id 불필요)
GROUP_ID_MAP = {"kpx_group_1": 0, "kpx_group_2": 1, "kpx_group_3": 2}
DEFAULT_CAT_PARAMS = {"iterations": 2000, "learning_rate": 0.05}
TAU = 0.70
T_SOFT = 0.035
MLP_LR, MLP_WD = 1e-3, 1e-4
GROUP_MLP_HP = {
    "kpx_group_1": {"hidden": (256, 256), "dropout": 0.10},
    "kpx_group_2": {"hidden": (384, 384), "dropout": 0.05},
    "kpx_group_3": {"hidden": (256, 256), "dropout": 0.15},  # v5 통합 설정값을 완전분리 시작점으로 사용
}


def make_answer_df(df):
    return df[["forecast_kst_dtm", *TARGET_COLS]].reset_index(drop=True)


def make_pred_df(df, pred_dict):
    out = df[["forecast_kst_dtm"]].reset_index(drop=True).copy()
    for col in TARGET_COLS:
        out[col] = np.clip(pred_dict[col], 0, CAPACITY_KWH[col])
    return out


def single_group_score(fold_valid, g, pred_kwh):
    actual = fold_valid[g].to_numpy(dtype=float)
    pred = np.asarray(pred_kwh, dtype=float)
    cap = CAPACITY_KWH[g]
    valid = actual >= cap * 0.10
    a, p = actual[valid], pred[valid]
    er = np.abs(p - a) / cap
    nmae = float(np.mean(er))
    price = np.select([er <= 0.06, er <= 0.08], [4.0, 3.0], default=0.0)
    ficr = float(np.sum(a * price) / np.sum(a * 4.0))
    return 0.5 * (1 - nmae) + 0.5 * ficr


def to_long_single(df, g, feature_cols):
    sub = df[df[g].notna()].copy()
    sub["utilization"] = sub[g] / CAPACITY_KWH[g]
    sub["actual_kwh"] = sub[g]
    return sub[["forecast_kst_dtm", "utilization", "actual_kwh"] + feature_cols].reset_index(drop=True)


def train_group_catboost(fold_train, g, feature_cols, seed=SEED, tau=TAU, early_stop_frac=0.15):
    long_df = to_long_single(fold_train, g, feature_cols).sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_df) * early_stop_frac)
    fit_rows, early_rows = long_df.iloc[:-n_early], long_df.iloc[-n_early:]
    model = CatBoostRegressor(loss_function=f"Quantile:alpha={tau}", random_seed=seed, verbose=False, **DEFAULT_CAT_PARAMS)
    model.fit(
        fit_rows[feature_cols], fit_rows["utilization"],
        eval_set=(early_rows[feature_cols], early_rows["utilization"]),
        sample_weight=fit_rows["actual_kwh"].to_numpy(), early_stopping_rounds=100, verbose=False,
    )
    return model


def predict_group_catboost(model, fold_valid, g, feature_cols):
    return model.predict(fold_valid[feature_cols]) * CAPACITY_KWH[g]


def build_mlp_features(df, feature_cols, mu=None, sd=None):
    num_x = df[feature_cols].fillna(0.0).to_numpy(dtype=np.float64)
    if mu is None:
        mu, sd = mlp_nn.fit_standardizer(num_x)
    num_x = mlp_nn.apply_standardizer(num_x, mu, sd).astype(np.float32)
    return num_x, mu, sd


def train_group_mlp(fold_train, g, feature_cols, seed=SEED, T_soft=T_SOFT, early_stop_frac=0.15, **hp_override):
    # 완전분리(그룹 1개만) 학습이라 그룹별 손실이 사실상 pooled와 동일해진다(n_groups=1, group_id 전부 0).
    hp = {**GROUP_MLP_HP[g], **hp_override}
    long_df = to_long_single(fold_train, g, feature_cols).sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_df) * early_stop_frac)
    fit_rows, early_rows = long_df.iloc[:-n_early], long_df.iloc[-n_early:]
    fit_X, mu, sd = build_mlp_features(fit_rows, feature_cols)
    early_X, _, _ = build_mlp_features(early_rows, feature_cols, mu=mu, sd=sd)
    fu = fit_rows["utilization"].to_numpy(np.float32); fk = fit_rows["actual_kwh"].to_numpy(np.float32)
    eu = early_rows["utilization"].to_numpy(np.float32); ek = early_rows["actual_kwh"].to_numpy(np.float32)
    fit_gid = np.zeros(len(fit_rows), dtype=np.int64)
    early_gid = np.zeros(len(early_rows), dtype=np.int64)
    model, _, _ = mlp_nn.train_mlp(
        fit_X, fu, fk, fu >= 0.10, early_X, eu, ek, eu >= 0.10,
        input_dim=fit_X.shape[1], seed=seed, T_soft=T_soft,
        hidden=hp["hidden"], dropout=hp["dropout"], lr=MLP_LR, weight_decay=MLP_WD,
        group_id=fit_gid, early_group_id=early_gid, n_groups=1,
    )
    return model, mu, sd


def predict_group_mlp_sep(model, fold_valid, g, feature_cols, mu, sd):
    x, _, _ = build_mlp_features(fold_valid, feature_cols, mu=mu, sd=sd)
    return mlp_nn.predict_mlp(model, x) * CAPACITY_KWH[g]


BLEND_GRID = [round(0.1 * i, 1) for i in range(11)]
print("완전분리 학습 헬퍼 준비 완료")

완전분리 학습 헬퍼 준비 완료


In [9]:
# 4-2. g3 안전장치: 완전분리로 먼저 시도해보고, 기존 50피처 단독학습 기준선(-0.0126)과 비교
def fold_frames_gen(df, fold):
    return fold_frames(df, fold)


def group3_standalone_check(feature_cols, tag):
    scores = []
    for fold in FOLDS:
        tr, va = fold_frames_gen(train_features_aug, fold)
        cat = train_group_catboost(tr, "kpx_group_3", feature_cols)
        cat_pred = predict_group_catboost(cat, va, "kpx_group_3", feature_cols)
        mlp, mu, sd = train_group_mlp(tr, "kpx_group_3", feature_cols)
        mlp_pred = predict_group_mlp_sep(mlp, va, "kpx_group_3", feature_cols, mu, sd)
        best = max(single_group_score(va, "kpx_group_3", (1 - w) * cat_pred + w * mlp_pred) for w in BLEND_GRID)
        scores.append(best)
    overall = float(np.mean(scores))
    print(f"[{tag}] g3 완전분리 단독 3-fold 그룹점수 = {overall:.4f} (fold별 {[round(s,4) for s in scores]})")
    return overall


G3_UNIFIED_BASELINE_OLD50 = 0.5957  # v4/v5 통합 g3 그룹점수 (05_tuning_3 7-2 UNI_GROUP_SCORE 기록)
g3_sep_new = group3_standalone_check(FEATURE_COLS_AUG, "확장피처")
print(f"\n비교 기준(통합, 기존 50피처): {G3_UNIFIED_BASELINE_OLD50}")
print(f"g3 완전분리(확장피처) vs 통합기준: {g3_sep_new - G3_UNIFIED_BASELINE_OLD50:+.4f}")
G3_FALLBACK_TO_UNIFIED = g3_sep_new < G3_UNIFIED_BASELINE_OLD50
print(f"\n-> g3 폴백 필요(통합으로 되돌림)?: {G3_FALLBACK_TO_UNIFIED}")

[확장피처] g3 완전분리 단독 3-fold 그룹점수 = 0.5754 (fold별 [0.5552, 0.5592, 0.6119])

비교 기준(통합, 기존 50피처): 0.5957
g3 완전분리(확장피처) vs 통합기준: -0.0203

-> g3 폴백 필요(통합으로 되돌림)?: True


## 5. 3-fold CV 비교 매트릭스 (A/B/C/D 디컨파운드)

| 구성 | 구조 | 피처 | 목적 |
|---|---|---|---|
| (A) v5 재현 | 하이브리드(g1/g2 전용+g3 통합) | 기존 50개 | 기준선(환경차 통제용 재계산) |
| (B) 완전분리+신규피처 | 3그룹 완전분리(g3는 4-2 결과로 조건부) | 확장 | **메인 실험** |
| (C) 완전분리+기존피처 | 3그룹 완전분리 | 기존 50개 | 구조 순효과(7-1 대조) |
| (D) 하이브리드+신규피처 | 하이브리드(v5 구조) | 확장 | 피처 순효과 |

Score = B−A(전체효과), B−D(구조 순효과), B−C(피처 순효과). fold3 병행 확인.

In [10]:
# 5-1. 헬퍼: 통합(pooled, group_id) CatBoost — 하이브리드 A/D용
GROUP_ID_CATEGORIES = [0, 1, 2]


def to_long_pooled(df, feature_cols):
    frames = []
    for g in TARGET_COLS:
        sub = df[df[g].notna()].copy()
        sub["group_id"] = GROUP_ID_MAP[g]
        sub["utilization"] = sub[g] / CAPACITY_KWH[g]
        sub["actual_kwh"] = sub[g]
        frames.append(sub[["forecast_kst_dtm", "group_id", "utilization", "actual_kwh"] + feature_cols])
    return pd.concat(frames, ignore_index=True)


def train_pooled_catboost(fold_train, feature_cols, seed=SEED, tau=TAU, early_stop_frac=0.15):
    features = feature_cols + ["group_id"]
    long_df = to_long_pooled(fold_train, feature_cols)
    long_df["group_id"] = pd.Categorical(long_df["group_id"], categories=GROUP_ID_CATEGORIES)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]
    model = CatBoostRegressor(loss_function=f"Quantile:alpha={tau}", random_seed=seed, verbose=False, **DEFAULT_CAT_PARAMS)
    model.fit(
        fit_rows[features], fit_rows["utilization"],
        eval_set=(early_rows[features], early_rows["utilization"]),
        cat_features=["group_id"], sample_weight=fit_rows["actual_kwh"].to_numpy(),
        early_stopping_rounds=100, verbose=False,
    )
    return model


def predict_pooled_catboost(model, fold_valid, g, feature_cols):
    features = feature_cols + ["group_id"]
    valid_g = fold_valid.copy()
    valid_g["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(valid_g), categories=GROUP_ID_CATEGORIES)
    return model.predict(valid_g[features]) * CAPACITY_KWH[g]


def train_unified_mlp(fold_train, feature_cols, seed=SEED, T_soft=T_SOFT, hidden=(256, 256), dropout=0.15, early_stop_frac=0.15):
    long_df = to_long_pooled(fold_train, feature_cols).sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_df) * early_stop_frac)
    fit_rows, early_rows = long_df.iloc[:-n_early], long_df.iloc[-n_early:]
    onehot_fit = pd.get_dummies(fit_rows["group_id"].astype(int), prefix="grp").reindex(columns=[f"grp_{i}" for i in range(3)], fill_value=0).to_numpy(np.float32)
    onehot_early = pd.get_dummies(early_rows["group_id"].astype(int), prefix="grp").reindex(columns=[f"grp_{i}" for i in range(3)], fill_value=0).to_numpy(np.float32)
    num_fit = fit_rows[feature_cols].fillna(0.0).to_numpy(dtype=np.float64)
    mu, sd = mlp_nn.fit_standardizer(num_fit)
    fit_X = np.concatenate([mlp_nn.apply_standardizer(num_fit, mu, sd).astype(np.float32), onehot_fit], axis=1)
    num_early = early_rows[feature_cols].fillna(0.0).to_numpy(dtype=np.float64)
    early_X = np.concatenate([mlp_nn.apply_standardizer(num_early, mu, sd).astype(np.float32), onehot_early], axis=1)
    fu = fit_rows["utilization"].to_numpy(np.float32); fk = fit_rows["actual_kwh"].to_numpy(np.float32)
    eu = early_rows["utilization"].to_numpy(np.float32); ek = early_rows["actual_kwh"].to_numpy(np.float32)
    model, _, _ = mlp_nn.train_mlp(
        fit_X, fu, fk, fu >= 0.10, early_X, eu, ek, eu >= 0.10,
        input_dim=fit_X.shape[1], seed=seed, T_soft=T_soft, hidden=hidden, dropout=dropout, lr=MLP_LR, weight_decay=MLP_WD,
    )
    return model, mu, sd


def predict_unified_mlp(model, fold_valid, g, feature_cols, mu, sd):
    valid_g = fold_valid.copy()
    onehot = pd.get_dummies(pd.Series([GROUP_ID_MAP[g]] * len(valid_g)), prefix="grp").reindex(columns=[f"grp_{i}" for i in range(3)], fill_value=0).to_numpy(np.float32)
    num_x = valid_g[feature_cols].fillna(0.0).to_numpy(dtype=np.float64)
    x = np.concatenate([mlp_nn.apply_standardizer(num_x, mu, sd).astype(np.float32), onehot], axis=1)
    return mlp_nn.predict_mlp(model, x) * CAPACITY_KWH[g]


def reopt_blend(cat_by_fold, mlp_by_fold, source_df):
    best_w, best_g = {}, {}
    for g in TARGET_COLS:
        top = (-1.0, None)
        for w in BLEND_GRID:
            fs = []
            for fold in FOLDS:
                _, va = fold_frames_gen(source_df, fold)
                pred = (1 - w) * cat_by_fold[fold["name"]][g] + w * mlp_by_fold[fold["name"]][g]
                fs.append(single_group_score(va, g, pred))
            s = float(np.mean(fs))
            if s > top[0]:
                top = (s, w)
        best_g[g], best_w[g] = top
    overall = float(np.mean([best_g[g] for g in TARGET_COLS]))
    return overall, best_w, best_g


def blend_fold3(cat_by_fold, mlp_by_fold, w, source_df):
    _, fv3 = fold_frames_gen(source_df, FOLDS[2])
    preds = {g: (1 - w[g]) * cat_by_fold["fold3"][g] + w[g] * mlp_by_fold["fold3"][g] for g in TARGET_COLS}
    s, _, _ = metric(make_answer_df(fv3), make_pred_df(fv3, preds))
    return s


print("A/D(하이브리드/통합) 헬퍼 준비 완료")

A/D(하이브리드/통합) 헬퍼 준비 완료


In [11]:
# 5-2. (A) v5 재현 -- 하이브리드(g1/g2 전용+g3 통합), 기존 50피처
def run_hybrid(source_df, feature_cols, seed=SEED):
    cat_by_fold, mlp_by_fold = {}, {}
    for fold in FOLDS:
        tr, va = fold_frames_gen(source_df, fold)
        cat = train_pooled_catboost(tr, feature_cols, seed=seed)
        cat_by_fold[fold["name"]] = {g: predict_pooled_catboost(cat, va, g, feature_cols) for g in TARGET_COLS}
        fold_mlp = {}
        for g in ["kpx_group_1", "kpx_group_2"]:
            m, mu, sd = train_group_mlp(tr, g, feature_cols, seed=seed)
            fold_mlp[g] = predict_group_mlp_sep(m, va, g, feature_cols, mu, sd)
        m3, mu3, sd3 = train_unified_mlp(tr, feature_cols, seed=seed, hidden=GROUP_MLP_HP["kpx_group_3"]["hidden"], dropout=GROUP_MLP_HP["kpx_group_3"]["dropout"])
        fold_mlp["kpx_group_3"] = predict_unified_mlp(m3, va, "kpx_group_3", feature_cols, mu3, sd3)
        mlp_by_fold[fold["name"]] = fold_mlp
    reopt, w, gscore = reopt_blend(cat_by_fold, mlp_by_fold, source_df)
    f3 = blend_fold3(cat_by_fold, mlp_by_fold, w, source_df)
    return {"reopt": reopt, "fold3": f3, "w": w, "gscore": gscore}


result_A = run_hybrid(train_features_aug, BASELINE_COLS)
print(f"(A) v5 재현: reopt={result_A['reopt']:.4f} fold3={result_A['fold3']:.4f} w={result_A['w']}")
print(f"   그룹점수: {result_A['gscore']}")

(A) v5 재현: reopt=0.6319 fold3=0.6532 w={'kpx_group_1': 0.3, 'kpx_group_2': 0.6, 'kpx_group_3': 0.8}
   그룹점수: {'kpx_group_1': 0.6454795954839211, 'kpx_group_2': 0.6544589441615433, 'kpx_group_3': 0.5956788415106394}


In [12]:
# 5-3. (D) 하이브리드 + 신규 확장피처 (구조는 A와 동일, 피처만 확장)
result_D = run_hybrid(train_features_aug, FEATURE_COLS_AUG)
print(f"(D) 하이브리드+신규피처: reopt={result_D['reopt']:.4f} fold3={result_D['fold3']:.4f} w={result_D['w']}")
print(f"   그룹점수: {result_D['gscore']}")
print(f"   피처 순효과(D-A): {result_D['reopt'] - result_A['reopt']:+.4f}")

(D) 하이브리드+신규피처: reopt=0.6278 fold3=0.6418 w={'kpx_group_1': 0.2, 'kpx_group_2': 0.5, 'kpx_group_3': 0.8}
   그룹점수: {'kpx_group_1': 0.6431724313798219, 'kpx_group_2': 0.651280582045906, 'kpx_group_3': 0.5890124700907529}
   피처 순효과(D-A): -0.0041


In [13]:
# 5-4. (C) 완전분리 + 기존 50피처 (구조 순효과, 7-1 대조)
def run_full_separation(source_df, feature_cols, seed=SEED, g3_mode="separate"):
    cat_by_fold, mlp_by_fold = {}, {}
    for fold in FOLDS:
        tr, va = fold_frames_gen(source_df, fold)
        fold_cat, fold_mlp = {}, {}
        for g in TARGET_COLS:
            if g == "kpx_group_3" and g3_mode == "unified":
                cat = train_pooled_catboost(tr, feature_cols, seed=seed)
                fold_cat[g] = predict_pooled_catboost(cat, va, g, feature_cols)
                m3, mu3, sd3 = train_unified_mlp(tr, feature_cols, seed=seed, hidden=GROUP_MLP_HP[g]["hidden"], dropout=GROUP_MLP_HP[g]["dropout"])
                fold_mlp[g] = predict_unified_mlp(m3, va, g, feature_cols, mu3, sd3)
            else:
                cat = train_group_catboost(tr, g, feature_cols, seed=seed)
                fold_cat[g] = predict_group_catboost(cat, va, g, feature_cols)
                m, mu, sd = train_group_mlp(tr, g, feature_cols, seed=seed)
                fold_mlp[g] = predict_group_mlp_sep(m, va, g, feature_cols, mu, sd)
        cat_by_fold[fold["name"]] = fold_cat
        mlp_by_fold[fold["name"]] = fold_mlp
    reopt, w, gscore = reopt_blend(cat_by_fold, mlp_by_fold, source_df)
    f3 = blend_fold3(cat_by_fold, mlp_by_fold, w, source_df)
    return {"reopt": reopt, "fold3": f3, "w": w, "gscore": gscore}


result_C = run_full_separation(train_features_aug, BASELINE_COLS, g3_mode="separate")
print(f"(C) 완전분리+기존피처: reopt={result_C['reopt']:.4f} fold3={result_C['fold3']:.4f} w={result_C['w']}")
print(f"   그룹점수: {result_C['gscore']}")
print(f"   구조 순효과 참고(C-A, 기존 7-1 기록 -0.0049 근방 재현 여부): {result_C['reopt'] - result_A['reopt']:+.4f}")

(C) 완전분리+기존피처: reopt=0.6249 fold3=0.6482 w={'kpx_group_1': 0.2, 'kpx_group_2': 0.5, 'kpx_group_3': 0.1}
   그룹점수: {'kpx_group_1': 0.6418587384105585, 'kpx_group_2': 0.6555114594278518, 'kpx_group_3': 0.5772611481877092}
   구조 순효과 참고(C-A, 기존 7-1 기록 -0.0049 근방 재현 여부): -0.0070


In [14]:
# 5-5. (B) 완전분리 + 신규 확장피처 (메인 실험, g3는 4-2 안전장치 결과로 조건부 결정)
g3_mode_for_B = "unified" if G3_FALLBACK_TO_UNIFIED else "separate"
print(f"g3 모드: {g3_mode_for_B} (4-2 안전장치 판정 기준)")

result_B = run_full_separation(train_features_aug, FEATURE_COLS_AUG, g3_mode=g3_mode_for_B)
print(f"(B) 완전분리+신규피처[g3={g3_mode_for_B}]: reopt={result_B['reopt']:.4f} fold3={result_B['fold3']:.4f} w={result_B['w']}")
print(f"   그룹점수: {result_B['gscore']}")

g3 모드: unified (4-2 안전장치 판정 기준)
(B) 완전분리+신규피처[g3=unified]: reopt=0.6258 fold3=0.6443 w={'kpx_group_1': 0.6, 'kpx_group_2': 0.3, 'kpx_group_3': 0.8}
   그룹점수: {'kpx_group_1': 0.6340744451498167, 'kpx_group_2': 0.6542795122348676, 'kpx_group_3': 0.5890124700907529}


In [15]:
# 5-6. 매트릭스 종합 + 효과 분해
summary_rows = [
    {"구성": "(A) v5 재현(하이브리드+기존50)", "reopt": result_A["reopt"], "fold3": result_A["fold3"]},
    {"구성": "(B) 완전분리+신규피처(메인)", "reopt": result_B["reopt"], "fold3": result_B["fold3"]},
    {"구성": "(C) 완전분리+기존50(구조효과)", "reopt": result_C["reopt"], "fold3": result_C["fold3"]},
    {"구성": "(D) 하이브리드+신규피처(피처효과)", "reopt": result_D["reopt"], "fold3": result_D["fold3"]},
]
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

전체효과 = result_B["reopt"] - result_A["reopt"]
구조순효과 = result_B["reopt"] - result_D["reopt"]
피처순효과 = result_B["reopt"] - result_C["reopt"]
상호작용 = result_B["reopt"] - result_C["reopt"] - result_D["reopt"] + result_A["reopt"]
print(f"\n전체효과(B-A)   = {전체효과:+.4f}")
print(f"구조 순효과(B-D) = {구조순효과:+.4f}  (신규피처 고정하고 구조만 바꾼 효과)")
print(f"피처 순효과(B-C) = {피처순효과:+.4f}  (완전분리 고정하고 피처만 바꾼 효과)")
print(f"상호작용(B-C-D+A) = {상호작용:+.4f}")
print(f"\nfold3: A={result_A['fold3']:.4f} B={result_B['fold3']:.4f} C={result_C['fold3']:.4f} D={result_D['fold3']:.4f}")
print("-> B가 A를 이기고 fold3도 유지하면 6절 seed 재검증으로.")

                   구성    reopt    fold3
(A) v5 재현(하이브리드+기존50) 0.631872 0.653217
    (B) 완전분리+신규피처(메인) 0.625789 0.644283
  (C) 완전분리+기존50(구조효과) 0.624877 0.648248
 (D) 하이브리드+신규피처(피처효과) 0.627822 0.641840

전체효과(B-A)   = -0.0061
구조 순효과(B-D) = -0.0020  (신규피처 고정하고 구조만 바꾼 효과)
피처 순효과(B-C) = +0.0009  (완전분리 고정하고 피처만 바꾼 효과)
상호작용(B-C-D+A) = +0.0050

fold3: A=0.6532 B=0.6443 C=0.6482 D=0.6418
-> B가 A를 이기고 fold3도 유지하면 6절 seed 재검증으로.


## 6. 조건부 seed 재검증

In [16]:
# 6-1. (B)가 (A)를 이기고 fold3 유지하면 seed 3개 재검증
if result_B["reopt"] <= result_A["reopt"] or result_B["fold3"] < result_A["fold3"]:
    print("(B)가 (A)를 못 이기거나 fold3 역행 -> seed 재검증 생략, 기각 판정.")
else:
    seed_scores = []
    for seed in [42, 7, 2024]:
        r = run_full_separation(train_features_aug, FEATURE_COLS_AUG, seed=seed, g3_mode=g3_mode_for_B)
        seed_scores.append(r["reopt"])
        print(f"seed={seed}: (B) reopt={r['reopt']:.4f}")
    m, sdv = float(np.mean(seed_scores)), float(np.std(seed_scores))
    a_seed_scores = []
    for seed in [42, 7, 2024]:
        r = run_hybrid(train_features_aug, BASELINE_COLS, seed=seed)
        a_seed_scores.append(r["reopt"])
    am, asdv = float(np.mean(a_seed_scores)), float(np.std(a_seed_scores))
    gap = m - am
    pooled = (sdv + asdv) / 2
    ratio = gap / pooled if pooled > 0 else float("nan")
    print(f"\n(B) seed평균={m:.4f}(std {sdv:.4f}) | (A) seed평균={am:.4f}(std {asdv:.4f})")
    print(f"gap={gap:+.4f} (평균표준편차의 {ratio:+.1f}배)")
    print("-> 여러 배(6배 기준 참고, 신호 성격도 함께 판단) 넘고 fold3 유지하면 채택 검토, train/inference 반영은 별도 논의.")

(B)가 (A)를 못 이기거나 fold3 역행 -> seed 재검증 생략, 기각 판정.


## 7. 종합 해석

**결과 (3-fold CV, 민석님 실행 — 사전 스크래치 검증과 소수점까지 일치, 측정 신뢰)**

| 구성 | reopt | fold3 |
|---|---:|---:|
| (A) v5 재현(하이브리드+기존50) | 0.6319 | 0.6532 |
| (B) 완전분리+신규피처(메인) | 0.6258 | 0.6443 |
| (C) 완전분리+기존50(구조효과) | 0.6249 | 0.6482 |
| (D) 하이브리드+신규피처(피처효과) | 0.6278 | 0.6418 |

**효과 분해**: 구조 순효과(B-D) = -0.0020, 피처 순효과(B-C) = +0.0009(노이즈 수준), 상호작용 = +0.0050. **(B)가 (A)를 -0.0061로 못 이기고 fold3도 -0.0089 역행 → 판정 기준(재최적·fold3 동시 유지) 미충족, 기각. seed 재검증은 6-1에서 조건 미달로 자동 생략.**

**해석**:
- **완전분리 구조 자체가 손해**(구조 순효과 -0.0020, C-A는 -0.0070으로 05_tuning_3 7-1의 -0.0049와 방향 일치·크기는 비슷한 급 — g3가 완전분리에서 크게 손해(-0.0203, 4-2)라는 게 재확인됨). 이 파이프라인(50~136개 curated 피처, 하이퍼파라미터 재탐색 없음) 규모에서는 그룹 간 정보 공유(하이브리드의 g3 통합)가 여전히 유리하다.
- **신규 확장 피처(65+21=86개)는 노이즈 수준**(피처 순효과 +0.0009, D-A는 -0.0041로 오히려 손해) — 09·25절(원본 풍속 rolling만)에 이어, 상층대기·구름·복사·강수·경계층까지 폭넓게 넓혀도 이 파이프라인 구조에서는 여전히 안 먹힌다는 것이 재확인됨.
- **왜 외부(0.63886)는 되고 우리는 안 됐는가**: 이번 빌드는 의도적으로 두 가지를 하지 않았다 — ① **하이퍼파라미터 재탐색**(외부는 그룹별로 τ/MLP epoch/hidden을 세밀 튜닝했으나, 우리는 v5 값을 그대로 시작점으로 씀 — 스크리닝 목적), ② **파워커브(`pc_pred`) 피처**(외부 importance 1위, 전체 gain의 15.7%를 이 피처 하나가 차지 — 우리는 17절 실패로 코드가 삭제돼 이번엔 재현하지 않음). 이 두 레버가 격차의 유력한 원인 후보로 남는다.

**결정**: 이 스크리닝(재탐색 없는 최초 시도)에서는 **기각**. train/inference 무변경, v5 유지(확정 최고 LB 0.629551). 다음 방향(하이퍼파라미터 재탐색 투자 여부, pc_pred 재구축 여부, 혹은 여기서 마무리)은 민석님과 상의해 결정.
